# Cuaderno de Laboratorio — Práctica 1: La Física del Aprendizaje Automático

**Asignatura:** Introducción al Aprendizaje Automático (3º Ing. Informática)  
**Curso:** 2025/2026  
**Alumno/a:** Alicia Muriel Fernández y Lucía Cañero Moslero  
**Fecha:** 26 de febrero de 2026

## Objetivo
Comprender la dinámica de la optimización mediante gradiente descendente y el impacto de los hiperparámetros en la convergencia.*  

Este cuaderno está pensado para generar las gráficas y escribir el análisis. La entrega final debe ser **PDF**.


## 1. Tarea 1 — La importancia de la tasa de aprendizaje (α)

### 1.1 Escenario A (αslow): convergencia lenta
- **Valor elegido:** α = 0.0001
- **Figura:** ![αslow](outputs/alpha_A_alpha_slow_a0.0001.png)

**Análisis:**
- **¿Por qué la curva baja tan lentamente?** 
Porque la tasa de aprendizaje ($\alpha$) es minúscula. El algoritmo da pasos demasiado pequeños en la dirección del gradiente en cada actualización de los pesos, por lo que apenas avanza hacia el mínimo en cada época.
- **¿Cuántas iteraciones estimas que harían falta para llegar cerca del mínimo?** 
Observando la pendiente casi plana en las primeras 120 épocas, se estima que harían falta miles de iteraciones (probablemente entre 5.000 y 10.000) para alcanzar la zona de estabilización.

### 1.2 Escenario B (αopt): “curva de codo”
- **Valor elegido:** α = 
- **Figura:**

![αopt](outputs/alpha_B_alpha_opt_a0.01.png)

**Análisis:**
- **Describe la “curva de codo” y qué te dice sobre estabilidad y rapidez:** 
La curva muestra un descenso abrupto en las primeras épocas (el brazo) y luego se aplana bruscamente manteniéndose estable (el antebrazo). Esto indica el equilibrio perfecto: el modelo es lo bastante rápido para llegar al mínimo del error en muy pocas iteraciones, pero lo bastante estable como para no rebotar una vez llega al fondo.

### 1.3 Escenario C (αosc): oscilación amortiguada
- **Valor elegido:** α = 0.1
- **Figura:** ![αosc](outputs/alpha_C_alpha_osc_a0.1.png)

**Análisis:**
- **¿Qué está ocurriendo “físicamente” con los pesos θ en el espacio de búsqueda?** 
Los pasos que da el algoritmo son demasiado grandes. Al intentar descender por el "valle" de la función de coste, el modelo se pasa de largo del punto mínimo y cruza a la ladera opuesta, rebotando de un lado a otro.
- **Explica por qué puede oscilar y aun así estabilizarse:** 
Consigue estabilizarse porque, a medida que se acerca al fondo del valle, la pendiente (el gradiente) es menor. Como la actualización de los pesos es el gradiente multiplicado por $\alpha$, al ser menor el gradiente, los pasos se van haciendo más pequeños progresivamente hasta que la oscilación se amortigua.

### 1.4 Escenario D (αfail): divergencia
- **Valor elegido:** α = 2.0
- **Figura:** ![αfail](outputs/alpha_D_alpha_fail_a2.0.png)

**Análisis:**
- **¿Por qué el error crece? Relaciónalo con “pasarse” del mínimo:** 
El tamaño del paso es tan desproporcionado que, al actualizar los pesos, el modelo salta a un punto de la función de coste más alto que el anterior. En cada iteración se aleja más del fondo del valle, haciendo que el error crezca exponencialmente (divergencia) en lugar de minimizarse.

## 2. Tarea 2 — El compromiso del Mini-batch

Comparativa utilizando **α = 0.01**:

### 2.1 Batch completo
![Batch completo](outputs/batch_Batch_completo_b500.png)

### 2.2 Mini-batch (32)
![Mini-batch](outputs/batch_Mini_batch_32_b32.png)

### 2.3 Estocástico puro (batch=1)
![Estocástico](outputs/batch_Estocastico_1_b1.png)

**Preguntas de reflexión:**
1. **¿Cuál de las tres curvas es más “ruidosa” y por qué?** La del Estocástico puro (batch=1) es la más ruidosa. Esto ocurre porque actualiza los pesos basándose en el gradiente de un solo dato cada vez. Cada dato tiene su propio ruido o desviación, lo que hace que el camino hacia el mínimo sea muy errático en comparación con hacer la media de varios datos.
2. **A nivel de tiempo de ejecución (CPU), ¿cuál ha sido más eficiente?** El Batch Completo es el más eficiente a nivel computacional puro por época. Al usar todos los datos a la vez, se aprovecha al máximo la computación vectorial (multiplicación de matrices) que está altamente optimizada en las CPU/GPU. El estocástico, por el contrario, obliga a hacer cientos de operaciones secuenciales pequeñas, perdiendo esa ventaja de paralelización. Sin embargo, el Mini-batch es el que ofrece el mejor equilibrio práctico entre eficiencia computacional y velocidad de convergencia.

In [ ]:
include("optimizacion_energia.jl")
# Si quieres el runner automático:
# include("experimentos_practica1.jl")


NameError: name 'include' is not defined

## 3. El reto — “Ajuste de Precisión” (criterio de parada)

Modificando el script para que el entrenamiento se detenga cuando:
$|J_t - J_{t-1}| < 10^{-5}$

- **Mejor combinación encontrada:** α = 0.01 ; batch = 32
- **Épocas hasta parar:** *(Nota: Escribe aquí el número de épocas que te indique la terminal al ejecutar el reto)*.
- **Figura:**
![Early stop](outputs/reto_early_stop.png)

**Conclusión:**
- **¿Qué combinación para antes con error aceptable?** La combinación de Mini-batch (32) con una tasa de aprendizaje óptima (0.01) logra converger rápidamente y activar el criterio de parada en pocas épocas.
- **¿Qué sacrificas (si algo) para conseguirlo?** Se sacrifica una minúscula fracción de precisión absoluta (llegar al mínimo global matemático exacto) a cambio de un ahorro masivo en tiempo de cómputo y recursos, evitando iteraciones innecesarias que apenas mejoran el modelo.

---

## 4. Conclusiones finales
A lo largo de esta práctica hemos comprobado empíricamente la "física" detrás del entrenamiento de modelos. Hemos visto que la tasa de aprendizaje ($\alpha$) es el regulador principal de la estabilidad: si es muy baja el modelo es ineficiente, y si es muy alta, el modelo se vuelve inestable y diverge. Por otro lado, el tamaño del batch actúa como un regulador del "ruido"; un batch grande aprovecha la computación vectorial para ser rápido en hardware, pero un mini-batch introduce el ruido justo para converger en menos épocas. Finalmente, implementar un criterio de parada (early stopping) ha demostrado ser una herramienta crítica en ingeniería para optimizar recursos, parando el entrenamiento justo cuando el beneficio marginal de seguir iterando es nulo.